# Lesson 6 — `nn.Module`, `Parameter` & Model Structure

## 学习目标

前面五节主要围绕 Tensor：

- Tensor Shape
- Broadcasting
- Matrix Multiplication
- Einsum
- Autograd

从这一节开始，我们正式进入：

> 如何把这些 Tensor 运算组织成可以训练的神经网络模型。

完成本节后，应能够：

1. 理解 `torch.Tensor`、`torch.nn.Parameter` 和 `torch.nn.Module` 的区别；
2. 理解为什么神经网络通常继承 `nn.Module`；
3. 理解 `__init__()` 和 `forward()` 的职责；
4. 理解 Parameter Registration；
5. 理解 `model.parameters()` 为什么能够找到模型参数；
6. 理解 `named_parameters()`；
7. 理解 Module 可以嵌套 Module；
8. 理解 `ModuleList`、`Sequential` 的基本作用；
9. 理解 `state_dict()`；
10. 理解 `train()` 与 `eval()` 的区别；
11. 理解 `.to(device)` 为什么能够移动整个模型；
12. 从零写出一个最小的 Linear Module；
13. 把之前学过的 Autograd 与 Module 连接起来。


## 1. 普通 Tensor

前面一直在使用：

`torch.Tensor`

Tensor 最重要的职责是：

> 存储数据，并参与 Tensor 运算。

例如：

$$
x.shape=(B,T,D)
$$

可以表示 Transformer 中一批 token representation。

普通 Tensor 可以：

- 进行矩阵乘法；
- broadcasting；
- reshape；
- autograd；
- 放到 CPU / GPU。

例如：

`requires_grad=True`

之后，它也可以参与梯度计算。

但是：

> 一个 Tensor 会参与梯度计算，并不意味着 PyTorch 自动把它当作“模型参数”。

这正是 `nn.Parameter` 出现的原因。


In [1]:
import torch

x = torch.randn(2, 3, requires_grad=True)

print(type(x))
print("requires_grad:", x.requires_grad)


<class 'torch.Tensor'>
requires_grad: True


## 2. `nn.Parameter`

`nn.Parameter` 本质上是一种特殊的 Tensor。

它最重要的额外语义是：

> 当 Parameter 被赋值为 `nn.Module` 的属性时，PyTorch 会自动把它注册为模型参数。

例如：

`self.weight = nn.Parameter(...)`

告诉 PyTorch：

> `weight` 不是普通中间数据，而是模型需要学习的参数。

这使得之后：

`model.parameters()`

可以自动找到它。

进一步，Optimizer 也可以通过：

`model.parameters()`

知道需要更新哪些 Tensor。


In [2]:
from torch import nn

tensor = torch.randn(3, 4)

parameter = nn.Parameter(torch.randn(3, 4))

print("tensor type:", type(tensor))
print("parameter type:", type(parameter))
print("tensor requires_grad:", tensor.requires_grad)
print("parameter requires_grad:", parameter.requires_grad)

tensor type: <class 'torch.Tensor'>
parameter type: <class 'torch.nn.parameter.Parameter'>
tensor requires_grad: False
parameter requires_grad: True


## 3. Tensor vs Parameter

两者都可以参与 Tensor 运算。

但在模型结构中语义不同。

### Tensor

通常表示：

- 输入；
- activation；
- intermediate result；
- 临时计算结果。

例如：

$$
X,\ Q,\ K,\ V,\ AttentionScores
$$

### Parameter

表示：

> 模型需要通过训练学习的 Tensor。

例如：

$$
W_Q
$$

$$
W_K
$$

$$
W_V
$$

$$
W_{FFN}
$$

以及 Embedding Matrix。

可以形成：

Tensor

→ 数据 / 中间状态

Parameter

→ 可学习模型状态


## 4. `nn.Module`

`nn.Module` 是 PyTorch 中组织神经网络的基本抽象。

可以把 Module 理解成一个容器：

Model / Layer

├── Parameters

├── Buffers

├── Submodules

└── Forward Computation

例如一个 Transformer Block 中可能包含：

TransformerBlock

├── RMSNorm

├── Attention

│   ├── Q Projection

│   ├── K Projection

│   ├── V Projection

│   └── Output Projection

├── RMSNorm

└── Feed-Forward Network

这些组件通常都会是：

`nn.Module`

因此大型模型实际上就是很多 Module 组成的一棵树。


## 5. 一个最小 `nn.Module`

定义 PyTorch Module 的典型结构：

`class MyModule(nn.Module):`

然后通常包含两个主要部分：

### `__init__()`

负责：

> 定义模型拥有什么。

例如：

- Parameters
- Submodules
- Configuration

### `forward()`

负责：

> 定义输入 Tensor 如何经过模型计算得到输出。

可以粗略理解：

`__init__`

→ Model Structure

`forward`

→ Model Computation


In [3]:
class Scale(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.scale * x


model = Scale()

x = torch.tensor(3.0)
y = model(x)

print("x:", x)
print("scale:", model.scale)
print("y:", y)

x: tensor(3.)
scale: Parameter containing:
tensor(1., requires_grad=True)
y: tensor(3., grad_fn=<MulBackward0>)


## 6. `super().__init__()`

定义 Module 时通常第一件事：

`super().__init__()`

它初始化 `nn.Module` 内部的基础设施。

这些基础设施负责管理：

- Parameter registration；
- Submodule registration；
- Buffer registration；
- Hooks；
- train / eval 状态；
- state_dict 等。

因此：

`nn.Module`

并不仅仅是一个普通 Python class。

它内部维护了一套模型状态管理机制。

正确模板：

`class MyModule(nn.Module):`

`    def __init__(self):`

`        super().__init__()`

不要省略这一行。


## 7. `forward()`

`forward()` 定义模型的 Forward Pass。

例如：

$$
y=ax
$$

那么：

`forward(x)`

负责：

1. 接收输入 Tensor；
2. 使用 Parameter；
3. 执行 Tensor Operations；
4. 返回输出 Tensor。

重要原则：

> `forward()` 应该描述模型的计算过程，而不是手动写 Backward。

例如：

`return self.scale * x`

PyTorch Autograd 会自动记录：

$$
scale
\rightarrow
multiplication
\rightarrow
y
$$

之后只需要：

`loss.backward()`

PyTorch 会自动得到：

$$
\frac{\partial L}{\partial scale}
$$


## 8. `model(x)` vs `model.forward(x)`

定义：

`forward()`

之后，我们实际使用模型时通常写：

`model(x)`

而不是：

`model.forward(x)`

原因是：

`nn.Module.__call__()`

会负责 Module 调用流程，然后内部再执行：

`forward()`。

可以粗略理解为：

`model(x)`

↓

`nn.Module.__call__(...)`

↓

PyTorch Module Infrastructure

↓

`forward(x)`

这种机制还涉及：

- hooks；
- autocast 等框架行为；
- Module 调用管理。

所以正常代码应该使用：

`model(x)`

而不是直接调用：

`model.forward(x)`。


In [4]:
model = Scale()

x = torch.tensor(5.0)
y = model(x)

print("model(x):", y)

model(x): tensor(5., grad_fn=<MulBackward0>)


## 9. Parameter Registration

这是本节最重要的机制之一。

假设：

`self.weight = nn.Parameter(...)`

因为 `weight`：

1. 是一个 `nn.Parameter`；
2. 被赋值给一个 `nn.Module` 属性。

PyTorch 会自动注册：

`weight`

之后：

`model.parameters()`

就能够找到它。

这称为：

> Parameter Registration

所以并不是变量名字叫 `weight` 就会被训练。

关键是：

`nn.Parameter`
+
`Module attribute`


In [5]:
class ExampleModule(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.weight = nn.Parameter(torch.randn(3, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight


model = ExampleModule()

for parameter in model.parameters():
    print(parameter.shape)

torch.Size([3, 4])


## 10. 普通 Tensor 与 Registration

考虑：

`self.weight = torch.randn(...)`

虽然它是 Tensor，但它不是：

`nn.Parameter`

因此不会自动成为 Module Parameter。

所以：

`model.parameters()`

不会返回这个普通 Tensor。

这说明：

> `requires_grad=True` 和“被注册为 Parameter”是两个不同概念。

一个 Tensor 可以：

`requires_grad=True`

但仍然不是模型正式注册的 Parameter。


In [6]:
class BadModule(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.weight = torch.randn(3, 4, requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight


model = BadModule()

print("requires_grad:", model.weight.requires_grad)
print("number of parameters:", len(list(model.parameters())))

requires_grad: True
number of parameters: 0


## 11. `named_parameters()`

`model.parameters()`

只能得到 Parameter Tensor。

但 debugging 时，我们通常还想知道：

> 这个 Parameter 叫什么名字？

可以使用：

`model.named_parameters()`

它返回：

`(name, parameter)`

例如：

`weight`

以及对应的 Tensor。

大型 Transformer 中会看到类似：

`blocks.0.attention.q_proj.weight`

`blocks.0.attention.k_proj.weight`

`blocks.0.ffn.w1.weight`

这些名称来自 Module 的嵌套结构。


In [7]:
model = ExampleModule()

for name, parameter in model.named_parameters():
    print("name:", name)
    print("shape:", parameter.shape)

name: weight
shape: torch.Size([3, 4])


## 12. Parameter 如何连接到 Autograd？

`nn.Parameter` 本质上仍然是 Tensor。

因此前一节学习的 Autograd 完全适用。

例如：

$$
W.shape=(D,H)
$$

Forward：

$$
Y=XW
$$

Loss：

$$
L
$$

调用：

`loss.backward()`

后：

$$
W.grad
=
\frac{\partial L}{\partial W}
$$

并且：

$$
W.grad.shape=W.shape
$$

所以：

Module

并没有替代 Autograd。

它只是把 Parameter 和模型结构组织起来。

真正梯度计算仍由 Autograd 完成。


In [8]:
model = ExampleModule()

x = torch.randn(2, 3)

output = model(x)

loss = output.pow(2).mean()
loss.backward()

print("weight shape:", model.weight.shape)
print("gradient shape:", model.weight.grad.shape)

weight shape: torch.Size([3, 4])
gradient shape: torch.Size([3, 4])


## 13. Submodule

真正的神经网络不会把所有 Parameter 都写在一个 class 中。

一个 Module 可以包含其他 Module。

例如：

Model

↓

Block

↓

Attention

↓

Projection

因此：

`nn.Module`

支持：

> Module 内嵌 Module。

当一个 Module 被赋值为另一个 Module 的属性时，PyTorch 会自动注册这个 Submodule。

这称为：

> Submodule Registration

于是父 Module：

`model.parameters()`

可以递归找到所有子 Module 中的 Parameters。


In [9]:
class ScalarMultiply(nn.Module):
    def __init__(self, initial_value: float) -> None:
        super().__init__()

        self.weight = nn.Parameter(torch.tensor(initial_value))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weight * x


class TwoStageModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.stage1 = ScalarMultiply(2.0)

        self.stage2 = ScalarMultiply(3.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stage1(x)
        x = self.stage2(x)

        return x


model = TwoStageModel()

x = torch.tensor(4.0)
y = model(x)

print("output:", y)

output: tensor(24., grad_fn=<MulBackward0>)


## 15. Recursive Parameter Discovery

`TwoStageModel`

本身没有直接定义：

`self.weight`

但是它包含：

`self.stage1`

和：

`self.stage2`

而这两个 Submodule 内部都有 Parameter。

PyTorch 会递归遍历 Module Tree。

因此：

`model.named_parameters()`

可以找到：

`stage1.weight`

以及：

`stage2.weight`

这正是 Transformer 可以被拆成：

Model

↓

Blocks

↓

Attention / FFN

↓

Linear / Norm

但仍然能够通过一个：

`model.parameters()`

找到所有参数的原因。


In [10]:
for name, parameter in model.named_parameters():
    print(name, parameter.shape, parameter.item())


stage1.weight torch.Size([]) 2.0
stage2.weight torch.Size([]) 3.0


## 16. Module Tree

PyTorch Model 可以理解成树结构。

例如：

TransformerLM

├── Embedding

├── Block 0

│   ├── RMSNorm

│   ├── Attention

│   │   ├── Q Projection

│   │   ├── K Projection

│   │   ├── V Projection

│   │   └── Output Projection

│   └── FeedForward

├── Block 1

├── ...

├── Final RMSNorm

└── LM Head

每一个节点通常都是：

`nn.Module`

叶子节点中通常包含：

`nn.Parameter`

因此：

`model.parameters()`

就是递归遍历整个 Module Tree 的参数。


## 17. `named_modules()`

除了 Parameter，还可以查看整个 Module Tree。

使用：

`model.named_modules()`

它会递归返回：

- Module 名称；
- Module 对象。

这是以后 debug Transformer 结构时非常有用的工具。

例如可以检查：

- Block 是否真的被注册；
- 某个 Attention Layer 是否存在；
- Module hierarchy 是否符合预期。


In [11]:
for name, module in model.named_modules():
    print(repr(name), "->", type(module).__name__)


'' -> TwoStageModel
'stage1' -> ScalarMultiply
'stage2' -> ScalarMultiply


## 18. 模型参数量

对于模型：

`model.parameters()`

每一个 Parameter 都是 Tensor。

Tensor 的元素数量：

`parameter.numel()`

因此总参数量：

$$
N
=
\sum_p
\operatorname{numel}(p)
$$

这就是之后计算 Transformer Parameter Count 的基础。

例如一个矩阵：

$$
W.shape=(D,H)
$$

参数量为：

$$
D\times H
$$

对于整个模型：

$$
N
=
\sum_{\text{all parameters}}
\text{parameter elements}
$$


In [12]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())

print("Total parameters:", total_parameters)

Total parameters: 2


## 19. Trainable Parameters

并非模型中的所有 Parameter 都必须训练。

Parameter 可以：

`requires_grad=False`

这通常称为：

> Frozen Parameter

所以可以区分：

### Total Parameters

所有 Parameter。

### Trainable Parameters

只统计：

`requires_grad=True`

的 Parameter。

计算：

$$
N_{trainable}
=
\sum_{p:\ requires\_grad}
\operatorname{numel}(p)
$$

后面做：

- Fine-tuning
- LoRA
- Frozen Embeddings

等场景时会经常使用这个概念。


In [13]:
total = sum(p.numel() for p in model.parameters())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total:", total)
print("Trainable:", trainable)

Total: 2
Trainable: 2


## 20. `state_dict()`

训练模型以后，需要保存模型状态。

PyTorch Module 提供：

`model.state_dict()`

它可以理解为：

> 模型可保存状态的字典。

其中通常包含：

- Parameters；
- Persistent Buffers。

key 对应 Module hierarchy 中的名字。

例如：

`stage1.weight`

`stage2.weight`

以后 Transformer checkpoint 的核心内容，就是类似这样的模型状态。

因此：

Module Structure

↓

Parameter Registration

↓

state_dict

↓

Checkpoint


In [14]:
state = model.state_dict()

for name, value in state.items():
    print(name, value)


stage1.weight tensor(2.)
stage2.weight tensor(3.)


## 21. Model Structure 与 Model State

可以区分两个概念。

### Model Structure

由 Python class 定义。

例如：

`class TransformerLM(nn.Module):`

规定：

- 有多少层；
- 每层包含什么；
- Forward 怎么计算。

### Model State

表示当前模型中的数值，例如：

- weights；
- biases；
- buffers。

`state_dict()` 保存的主要是：

> State

而不是完整 Python 模型代码。

所以重新加载模型通常需要：

1. 先重新创建同样结构的 Module；
2. 再加载 state_dict。


In [15]:
original_model = TwoStageModel()

state = original_model.state_dict()

new_model = TwoStageModel()
new_model.load_state_dict(state)

for name, parameter in new_model.named_parameters():
    print(name, parameter)


stage1.weight Parameter containing:
tensor(2., requires_grad=True)
stage2.weight Parameter containing:
tensor(3., requires_grad=True)


## 22. Parameter vs Buffer

Module 中并不是所有模型状态都需要训练。

有些 Tensor：

- 属于模型状态；
- 应该跟随模型移动到设备；
- 应该保存到 state_dict；
- 但不应该被 Optimizer 更新。

这种 Tensor 可以注册成：

> Buffer

典型概念上可以理解：

Parameter

→ Learnable State

Buffer

→ Non-learnable Model State

Buffer 使用：

`register_buffer()`

注册。


In [16]:
class ModuleWithBuffer(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.weight = nn.Parameter(torch.tensor(2.0))

        self.register_buffer("scale", torch.tensor(3.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.weight * self.scale


model_buffer = ModuleWithBuffer()

print("Parameters:")

for name, parameter in model_buffer.named_parameters():
    print(name, parameter)
print()
print("Buffers:")

for name, buffer in model_buffer.named_buffers():
    print(name, buffer)


Parameters:
weight Parameter containing:
tensor(2., requires_grad=True)

Buffers:
scale tensor(3.)


## 23. Parameter vs Buffer

可以建立下面的区分：

### Parameter

需要：

- 出现在 `model.parameters()`；
- 通常需要 Gradient；
- 被 Optimizer 更新；
- 出现在 state_dict。

### Buffer

需要：

- 属于 Module；
- 跟随设备迁移；
- 通常不需要 Gradient；
- 不出现在 `model.parameters()`；
- persistent buffer 会出现在 state_dict。

### 普通 Tensor Attribute

如果只是：

`self.x = torch.tensor(...)`

则它只是普通 Python attribute 中保存的 Tensor。

它不自动具有 Parameter / Buffer 的完整 Module 管理语义。


## 24. Module 与 Device

前面使用 Tensor 时：

`x = x.to(device)`

一个大型模型可能拥有数百甚至数千个 Parameter。

不可能手动逐个写：

`weight1.to(device)`

`weight2.to(device)`

`weight3.to(device)`

...

Module 提供：

`model.to(device)`

它会递归处理已注册的：

- Parameters；
- Buffers。

这再次说明：

> Registration 不只是为了 `model.parameters()`。

它还支撑模型状态的统一管理。


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_device = ModuleWithBuffer().to(device)

print("parameter device:", model_device.weight.device)
print("buffer device:", model_device.scale.device)

parameter device: cuda:0
buffer device: cuda:0


## 25. Model Device 与 Input Device

如果模型 Parameter 位于：

`cuda`

而输入位于：

`cpu`

很多运算会报 device mismatch。

因此通常要保证：

$$
device(X)=device(ModelParameters)
$$

例如：

`model = model.to(device)`

之后：

`x = x.to(device)`

然后：

`y = model(x)`

这也是之后训练 Transformer 时非常常见的基本结构。


In [18]:
x = torch.tensor(2.0, device=device)

y = model_device(x)

print("input device:", x.device)
print("output device:", y.device)

input device: cuda:0
output device: cuda:0


## 26. Training Mode 与 Evaluation Mode

每个 `nn.Module` 都有：

`training`

状态。

调用：

`model.train()`

会把模型切换到 Training Mode。

调用：

`model.eval()`

会切换到 Evaluation Mode。

注意：

> `eval()` 不等于关闭 Gradient。

它们控制的是某些 Module 在训练和推理时的不同行为。

典型例子包括：

- Dropout；
- BatchNorm。

Transformer 中如果使用 Dropout，这一点就非常重要。

所以需要区分：

### `model.eval()`

控制 Module 行为模式。

### `torch.no_grad()`

控制 Autograd 是否记录梯度。


In [19]:
model_mode = TwoStageModel()

print("initial:", model_mode.training)

model_mode.eval()

print("after eval:", model_mode.training)

model_mode.train()

print("after train:", model_mode.training)

initial: True
after eval: False
after train: True


## 27. `eval()` 和 Autograd 是两个不同系统

这是常见误区。

调用：

`model.eval()`

之后 Parameter 仍然可能：

`requires_grad=True`

如果正常执行 Forward，Autograd 仍然可以构建 graph。

所以推理通常会组合：

`model.eval()`

和：

`torch.no_grad()`

概念上：

`model.eval()`

→ 改变 Module Behavior

`torch.no_grad()`

→ 关闭当前代码区域的 Gradient Tracking


In [20]:
model_mode = Scale()
model_mode.eval()

x = torch.tensor(2.0, requires_grad=True)

y = model_mode(x)

print("model.training:", model_mode.training)
print("y.requires_grad:", y.requires_grad)

model.training: False
y.requires_grad: True


## 28. 打印 Module

直接：

`print(model)`

PyTorch 会显示注册的 Submodules。

对于大型网络，这可以帮助快速理解模型结构。

例如之后可能看到：

TransformerLM(

  (token_embedding): Embedding(...)

  (blocks): ModuleList(...)

  (norm): RMSNorm(...)

  (lm_head): Linear(...)

)

这实际上就是 Module Tree 的可读表示。

因此遇到陌生模型时，一个很简单的第一步就是：

`print(model)`


In [21]:
model = TwoStageModel()

print(model)

TwoStageModel(
  (stage1): ScalarMultiply()
  (stage2): ScalarMultiply()
)


## 29. 多层网络中的一个重要陷阱

假设需要很多层：

Block 0

Block 1

Block 2

...

一种自然的 Python 写法可能是：

`self.layers = [layer1, layer2, layer3]`

但是：

> 普通 Python list 本身不会按照 Module 容器的方式注册其中的 Submodules。

这可能导致：

`model.parameters()`

找不到里面的参数。

因此 PyTorch 为 Module 集合提供：

`nn.ModuleList`

之后写 Transformer Blocks 时：

`self.layers = nn.ModuleList([...])`

会非常常见。


In [22]:
class BadStack(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.layers = [Scale(), Scale()]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)

        return x


class GoodStack(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.layers = nn.ModuleList([Scale(), Scale()])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)

        return x


bad_model = BadStack()
good_model = GoodStack()

print("BadStack parameters:", len(list(bad_model.parameters())))
print("GoodStack parameters:", len(list(good_model.parameters())))

BadStack parameters: 0
GoodStack parameters: 2


## 30. Transformer Blocks 与 `ModuleList`

假设 Transformer 有：

$$
L
$$

个 Block。

逻辑结构：

Transformer

↓

Block 0

↓

Block 1

↓

...

↓

Block L-1

代码通常需要保存多个相同结构的 Module。

一种典型形式：

`self.blocks = nn.ModuleList([...])`

Forward：

`for block in self.blocks:`

`    x = block(x)`

这样：

1. Python 可以方便循环；
2. 每个 Block 仍然被 PyTorch 正确注册；
3. `model.parameters()` 可以递归找到所有 Block 参数；
4. `state_dict()` 可以保存所有 Block；
5. `model.to(device)` 可以移动所有 Block。

这就是 PyTorch Module System 对 Transformer 结构的重要作用。


## 31. `nn.ParameterList`

和 `ModuleList` 类似，如果需要保存：

> 多个独立 Parameter，而不是多个 Module

可以使用：

`nn.ParameterList`

例如：

`self.weights = nn.ParameterList([...])`

区别：

### `ModuleList`

内部保存：

`nn.Module`

### `ParameterList`

内部保存：

`nn.Parameter`

当前阶段重点记住 `ModuleList` 即可。

真正 Transformer 实现中，我们通常更多依赖：

- Module；
- Parameter；
- ModuleList。


## 32. 同一个 Parameter 可以被重复使用

Module 的一个重要特点是：

> Parameter 是一个真实 Tensor 对象，可以在 Forward 中被多次使用。

例如：

$$
y=W x + W z
$$

虽然 $W$ 被用了两次，但仍然只有一份 Parameter。

Backward 时，根据 Chain Rule：

$$
\frac{\partial L}{\partial W}
$$

会包含两条使用路径贡献的总和。

这和前面学习的 Gradient Accumulation / Computational Graph 是一致的。

未来语言模型中：

> Weight Tying

就是 Parameter Sharing 的一个重要例子。


## 33. Module、Autograd、Optimizer 的职责分工

这里很容易混淆。

### `nn.Module`

负责：

- 组织 Parameters；
- 组织 Submodules；
- 定义 Forward；
- 管理模型状态。

### Autograd

负责：

$$
L
\rightarrow
\nabla_\theta L
$$

即计算 gradient。

### Optimizer

负责：

$$
\theta
\leftarrow
\theta
+
\text{update}
$$

即使用 gradient 更新 Parameter。

所以：

`loss.backward()`

不会更新 Parameter。

它只计算：

`.grad`

而：

`model(x)`

也不会更新 Parameter。

真正更新通常由之后的：

`optimizer.step()`

完成。

完整职责：

Module

→ What is the model?

Autograd

→ What are the gradients?

Optimizer

→ How are parameters updated?


## 34. 第一次把 Module 与 Autograd 连接起来

暂时不用 PyTorch Optimizer。

继续使用 Lesson 5 的手动 Gradient Descent。

模型：

$$
\hat y=wx
$$

但是现在把：

$$
w
$$

封装在：

`nn.Module`

中。

这样可以观察完整流程：

Module

↓

Parameter

↓

Forward

↓

Loss

↓

Backward

↓

Parameter.grad

↓

Manual Update


In [23]:
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.weight = nn.Parameter(torch.tensor(0.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weight * x


model = SimpleModel()

x = torch.tensor(2.0)
target = torch.tensor(6.0)

learning_rate = 0.1


for step in range(10):
    model.weight.grad = None

    prediction = model(x)

    loss = (prediction - target) ** 2

    loss.backward()

    with torch.no_grad():
        model.weight -= learning_rate * model.weight.grad

    print(
        f"step={step:2d}",
        f"weight={model.weight.item():.4f}",
        f"loss={loss.item():.4f}",
    )


step= 0 weight=2.4000 loss=36.0000
step= 1 weight=2.8800 loss=1.4400
step= 2 weight=2.9760 loss=0.0576
step= 3 weight=2.9952 loss=0.0023
step= 4 weight=2.9990 loss=0.0001
step= 5 weight=2.9998 loss=0.0000
step= 6 weight=3.0000 loss=0.0000
step= 7 weight=3.0000 loss=0.0000
step= 8 weight=3.0000 loss=0.0000
step= 9 weight=3.0000 loss=0.0000


## 35. 为什么 `model.parameters()` 很重要？

刚才模型只有一个 Parameter：

`model.weight`

所以可以手动更新。

但是大型模型可能有数十亿 Parameter elements。

代码不可能写：

`model.weight1 -= ...`

`model.weight2 -= ...`

...

而：

`model.parameters()`

提供统一接口。

因此可以：

`for parameter in model.parameters():`

对所有 Parameter 执行统一操作。

这也是 Optimizer 接收：

`model.parameters()`

的根本原因。


In [24]:
for parameter in model.parameters():
    print("value:", parameter.item())
    print("gradient:", parameter.grad.item())


value: 2.999999761581421
gradient: -1.1444091796875e-05


## 36. 两参数模型

考虑：

$$
\hat y=wx+b
$$

现在模型包含：

$$
w
$$

和：

$$
b
$$

两个 Parameters。

Module 会自动注册：

`weight`

以及：

`bias`

Backward 后：

$$
weight.grad
=
\frac{\partial L}{\partial w}
$$

$$
bias.grad
=
\frac{\partial L}{\partial b}
$$

这已经非常接近下一节要自己实现的 Linear Layer。


In [25]:
class AffineModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.weight = nn.Parameter(torch.tensor(1.0))

        self.bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weight * x + self.bias


model = AffineModel()

x = torch.tensor(2.0)
target = torch.tensor(7.0)

prediction = model(x)

loss = (prediction - target) ** 2
loss.backward()

for name, parameter in model.named_parameters():
    print(name, "value =", parameter.item(), "grad =", parameter.grad.item())


weight value = 1.0 grad = -20.0
bias value = 0.0 grad = -10.0


## 37. Module 也要继续保持 Shape Thinking

不要因为进入 `nn.Module` 就停止追踪 shape。

未来我们的 Module 应该始终能够回答：

### 输入 Shape

$$
X.shape=?
$$

### Parameter Shape

$$
W.shape=?
$$

### 输出 Shape

$$
Y.shape=?
$$

### Gradient Shape

$$
W.grad.shape=?
$$

例如未来 Linear：

$$
X.shape=(B,T,D_{in})
$$

$$
W.shape=(D_{out},D_{in})
$$

经过 Linear：

$$
Y.shape=(B,T,D_{out})
$$

而：

$$
W.grad.shape=W.shape
$$

所以 Module Thinking 和 Shape Thinking 必须结合。


## 38. Structure vs Computation

一个很重要的代码设计原则。

### `__init__`

定义长期存在的模型组件。

例如：

- Parameters；
- Layers；
- Submodules；
- Buffers；
- Configuration。

### `forward`

定义一次 Forward Pass 中发生的计算。

例如：

- Matrix Multiplication；
- Attention；
- Normalization；
- Residual Connection。

不要在每次 `forward()` 中重新创建需要训练的 Parameter。

错误思想：

Forward

↓

每次新建一个 Weight

↓

计算输出

这样每次调用模型都会出现新的 Parameter。

Trainable Parameter 应该通常在：

`__init__`

中创建并注册。


## 39. 常见错误

### 错误 1：忘记继承 `nn.Module`

PyTorch 模型通常应该：

`class Model(nn.Module):`

---

### 错误 2：忘记 `super().__init__()`

这会破坏 Module 内部注册机制。

---

### 错误 3：把 Trainable Weight 写成普通 Tensor

错误：

`self.weight = torch.randn(...)`

即使：

`requires_grad=True`

它也不自动成为注册 Parameter。

应该：

`self.weight = nn.Parameter(...)`

---

### 错误 4：直接调用 `model.forward(x)`

正常使用应该：

`model(x)`

---

### 错误 5：认为 `nn.Module` 会自动更新 Parameter

Module 只组织模型。

Parameter 更新由 Optimizer 完成。

---

### 错误 6：认为 `loss.backward()` 会修改 Weight 数值

Backward 主要计算：

`parameter.grad`

并不会自动执行 Gradient Descent。

---

### 错误 7：使用普通 Python List 保存 Submodules

如果希望 PyTorch 注册多个层，应该考虑：

`nn.ModuleList`

---

### 错误 8：把 Buffer 当成 Parameter

Buffer 属于模型状态，但通常不参与 Optimizer 更新。

---

### 错误 9：认为 `model.eval()` 会关闭 Gradient

不会。

`eval()`

控制 Module Mode。

`no_grad()`

控制 Gradient Tracking。

---

### 错误 10：只看模型代码，不检查 Parameter

遇到问题时应该学会使用：

`model.named_parameters()`

`model.named_modules()`

`model.state_dict()`

检查模型结构。

---

### 错误 11：忘记输入与模型设备一致

如果：

Model → CUDA

Input → CPU

通常会产生 device mismatch。

---

### 错误 12：在 `forward()` 中创建新的 Trainable Parameter

Parameter 一般应该在：

`__init__()`

中定义并注册。


## 本节总结

### Rule 1：Tensor

Tensor 是数据与计算的基本单位。

---

### Rule 2：Parameter

`nn.Parameter`

是具有：

> Trainable Model Parameter

语义的 Tensor。

---

### Rule 3：Parameter Registration

当：

`nn.Parameter`

被赋值为 Module attribute：

`self.weight = nn.Parameter(...)`

PyTorch 会自动注册。

---

### Rule 4：Module

`nn.Module`

负责：

- Parameters；
- Submodules；
- Buffers；
- Forward Computation；
- Model State。

---

### Rule 5：`__init__`

定义：

> 模型拥有什么。

---

### Rule 6：`forward`

定义：

> 输入如何变成输出。

---

### Rule 7：调用 Module

正常使用：

`model(x)`

而不是：

`model.forward(x)`。

---

### Rule 8：Parameter Discovery

`model.parameters()`

返回所有注册 Parameter。

`model.named_parameters()`

同时返回 Parameter 名称。

---

### Rule 9：Submodule Registration

Module 可以包含 Module。

父 Module 会递归管理子 Module。

---

### Rule 10：ModuleList

多个 Submodule 通常应该使用：

`nn.ModuleList`

而不是普通 Python list。

---

### Rule 11：Buffer

Parameter：

→ Learnable State

Buffer：

→ Non-learnable Model State

---

### Rule 12：state_dict

`state_dict()`

保存 Module 的模型状态。

---

### Rule 13：Device

`model.to(device)`

会递归移动注册的 Parameters 和 Buffers。

---

### Rule 14：Train / Eval

`model.train()`

和：

`model.eval()`

控制 Module 的运行模式。

它与 Autograd 是不同机制。

---

### Rule 15：三者职责

`nn.Module`

→ Organize the model

Autograd

→ Compute gradients

Optimizer

→ Update parameters

---

### Rule 16：Module 与 Shape Thinking 必须结合

每个 Module 都应该能够回答：

$$
Input\ Shape
$$

$$
Parameter\ Shape
$$

$$
Output\ Shape
$$

$$
Gradient\ Shape
$$


## Module Debug Checklist

以后遇到：

> 为什么模型参数不训练？

可以按以下顺序检查。

### 1. 这个对象是不是 `nn.Module`？

检查：

`isinstance(model, nn.Module)`

---

### 2. Parameter 是否真的是 `nn.Parameter`？

检查：

`type(parameter)`

---

### 3. Parameter 是否注册成功？

运行：

`model.named_parameters()`

---

### 4. Parameter 是否：

`requires_grad=True`

---

### 5. Parameter 是否参与 Forward？

必须存在：

Parameter

↓

Forward

↓

Loss

这条 Graph。

---

### 6. 是否调用：

`loss.backward()`

---

### 7. `.grad` 是否存在？

检查：

`parameter.grad`

---

### 8. Submodule 是否正确注册？

多个 Module 是否错误地放进普通：

`list`

而不是：

`ModuleList`

？

---

### 9. Parameter 与输入是否在同一 Device？

检查：

`parameter.device`

和：

`x.device`

---

### 10. 是否错误 Freeze 了 Parameter？

检查：

`parameter.requires_grad`

---

### 11. 是否错误使用：

`detach()`

或：

`torch.no_grad()`

切断了 Graph？

---

### 12. 参数是否被 Optimizer 管理？

后面学习 Optimizer 后，还要检查：

> Optimizer 是否真的拿到了这个 Parameter？
